# Week 3 - Build the retriever

Picking up from parse_clean.ipynb. Input is topics_english_all.json (1016 English topics, cleaned).

Goal for this week: get a working search tool. Given a patient question, return the most relevant
bits of MedlinePlus. No LLM yet - that's next week. Retrieval quality caps everything downstream,
so if the right passage doesn't come back here, no generator can fix it later.

In [92]:
import json
from collections import Counter

import chromadb
from sentence_transformers import SentenceTransformer

In [93]:
# keeping settings together up here so the ablations later (different embedding
# model, different k) are a one line change instead of hunting through cells
TOPICS_JSON = 'topics_english_all.json'
DB_DIR = 'chroma_db'
COLLECTION = 'medlineplus_english'
EMBED_MODEL = 'BAAI/bge-small-en-v1.5'

MIN_CHUNK_CHARS = 200   # paragraphs shorter than this get merged forward
TOP_K = 5
MAX_PER_TOPIC = 2       # see the dedup section - stops one topic taking every slot

In [94]:
with open(TOPICS_JSON, encoding='utf-8') as f:
    topics = json.load(f)

print("topics loaded:", len(topics))
print("fields:", list(topics[0].keys()))

topics loaded: 1016
fields: ['id', 'title', 'url', 'summary', 'word_count', 'groups', 'also_called', 'sites']


## Sorting out the group labels first

Noticed this while looking at the data: a lot of topics sit in more than one group, and the first
one listed isn't always the medical one. Heart Failure comes back as ['Older Adults', 'Blood, Heart
and Circulation'] - so if I just take groups[0] everywhere, Heart Failure gets filed under Older
Adults. That would mess up the per category error analysis later, so worth checking how common it is.

In [95]:
multi = [t for t in topics if len(t['groups']) > 1]
print(f"topics in more than one group: {len(multi)} / {len(topics)}")

# these are audience buckets, not body systems / conditions
demographic_groups = {'Older Adults', 'Women', 'Men', 'Children and Teenagers', 'Population Groups'}

demo_first = [t for t in topics if t['groups'] and t['groups'][0] in demographic_groups]
print(f"topics whose FIRST group is a demographic bucket: {len(demo_first)}")
print("examples:")
for t in demo_first[:5]:
    print("  ", t['title'], ":", t['groups'])

topics in more than one group: 577 / 1016
topics whose FIRST group is a demographic bucket: 162
examples:
   Acute Flaccid Myelitis : ['Children and Teenagers', 'Infections', 'Brain and Nerves']
   Adenoids : ['Children and Teenagers', 'Ear, Nose and Throat']
   Alzheimer's Caregivers : ['Older Adults', 'Brain and Nerves', 'Social/Family Issues']
   Angina : ['Older Adults', 'Blood, Heart and Circulation']
   Assisted Living : ['Older Adults', 'Health System']


In [96]:
def primary_group(topic):
    for g in topic['groups']:
        if g not in demographic_groups:
            return g
    return topic['groups'][0] if topic['groups'] else 'Ungrouped'


# quick before/after so I can see what this actually changed
changed = [t for t in topics if t['groups'] and primary_group(t) != t['groups'][0]]
print(f"topics reassigned to a different primary group: {len(changed)}")
for t in changed[:5]:
    print(f"  {t['title']}: {t['groups'][0]} -> {primary_group(t)}")

print("\nprimary group distribution (top 10):")
for g, n in Counter(primary_group(t) for t in topics).most_common(10):
    print(f"  {n:3d}  {g}")

topics reassigned to a different primary group: 146
  Acute Flaccid Myelitis: Children and Teenagers -> Infections
  Adenoids: Children and Teenagers -> Ear, Nose and Throat
  Alzheimer's Caregivers: Older Adults -> Brain and Nerves
  Angina: Older Adults -> Blood, Heart and Circulation
  Assisted Living: Older Adults -> Health System

primary group distribution (top 10):
  111  Infections
   84  Blood, Heart and Circulation
   75  Bones, Joints and Muscles
   70  Brain and Nerves
   69  Digestive System
   66  Cancers
   42  Mental Health and Behavior
   38  Food and Nutrition
   34  Skin, Hair and Nails
   31  Pregnancy and Reproduction


In [97]:
demo_only = [t for t in topics if t['groups'] and all(g in demographic_groups for g in t['groups'])]
print(f"demographic-only topics: {len(demo_only)}")
for t in demo_only:
    print("  ", t['title'], "->", t['groups'])

demographic-only topics: 16
   Baby Health Checkup -> ['Children and Teenagers']
   Child Development -> ['Children and Teenagers']
   Children's Health -> ['Children and Teenagers', 'Population Groups']
   Common Infant and Newborn Problems -> ['Children and Teenagers']
   Infant and Newborn Care -> ['Children and Teenagers']
   Infant and Newborn Development -> ['Children and Teenagers']
   Men's Health -> ['Men', 'Population Groups']
   Older Adult Health -> ['Older Adults', 'Population Groups']
   School Health -> ['Children and Teenagers']
   Sudden Infant Death Syndrome -> ['Children and Teenagers']
   Teen Health -> ['Children and Teenagers', 'Population Groups']
   Toddler Development -> ['Children and Teenagers']
   Toddler Health -> ['Children and Teenagers']
   Uncommon Infant and Newborn Problems -> ['Children and Teenagers']
   Veterans and Military Family Health -> ['Population Groups']
   Women's Health -> ['Women', 'Population Groups']


## Chunking

The summaries are too long to embed whole - a 1000 word topic turned into one vector loses all the
detail. So they get split into chunks.

Going with paragraph splits rather than fixed character windows. Two reasons: fixed windows cut
sentences in half wherever the counter happens to land, and the cleaning notebook already preserved
the paragraph breaks so the structure is sitting right there. Very short paragraphs (a heading on
its own, one bullet) get merged forward so nothing meaningless ends up in the index.

Each chunk also gets the title and the also-called synonyms stuck on the front. Without that, a
chunk reading "Symptoms include increased thirst" has nothing in it saying diabetes, so it won't
match a diabetes query. The synonyms help with the opposite problem - people search "high blood
pressure" but the topic might be titled Hypertension.

In [98]:
def chunk_topic(topic):
    paragraphs = [p for p in topic['summary'].split('\n\n') if p.strip()]

    chunks = []
    buffer = ''
    for p in paragraphs:
        buffer = (buffer + '\n\n' + p).strip() if buffer else p
        if len(buffer) >= MIN_CHUNK_CHARS:
            chunks.append(buffer)
            buffer = '' if buffer == p else p   # last paragraph feeds into next chunk

    # whatever is left over at the end is too short to stand alone, so glue it
    # onto the previous chunk rather than indexing a fragment
    if buffer:
        if chunks:
            chunks[-1] = chunks[-1] + '\n\n' + buffer
        else:
            chunks.append(buffer)

    prefix = topic['title']
    if topic['also_called']:
        # capping at 4 synonyms, some topics have a lot and it drowns out the actual text
        prefix = prefix + ' (also called: ' + ', '.join(topic['also_called'][:4]) + ')'

    return [prefix + ': ' + c for c in chunks]

In [99]:
# sanity check the chunking before embedding anything
chunks_by_topic = {t['id']: chunk_topic(t) for t in topics}

counts = [len(c) for c in chunks_by_topic.values()]
sizes = sorted(len(ch) for cl in chunks_by_topic.values() for ch in cl)

print(f"total chunks: {sum(counts)}  ({sum(counts)/len(topics):.1f} per topic on average)")
print(f"chunks per topic: min {min(counts)}, max {max(counts)}")
print(f"chunk size in chars: min {sizes[0]}, median {sizes[len(sizes)//2]}, max {sizes[-1]}")

total chunks: 10337  (10.2 per topic on average)
chunks per topic: min 1, max 40
chunk size in chars: min 194, median 343, max 1213


In [100]:
long_chunks = [ch for cl in chunks_by_topic.values() for ch in cl if len(ch) > 1000]
print(f"chunks over 1000 chars: {len(long_chunks)} / {sum(counts)}")

chunks over 1000 chars: 26 / 10337


In [101]:
#view one topic properly
example = topics[0]
print("TOPIC:", example['title'], "|", example['word_count'], "words")
for i, ch in enumerate(chunk_topic(example)):
    print(f"\n--- chunk {i} ({len(ch)} chars) ---")
    print(ch[:250].replace('\n', ' '))

TOPIC: A1C | 199 words

--- chunk 0 (464 chars) ---
A1C (also called: Glycohemoglobin, HbA1C, Hemoglobin A1C test): A1C is a blood test for type 2 diabetes and prediabetes . It measures your average blood glucose, or blood sugar , level over the past 3 months. Doctors may use the A1C alone or in combi

--- chunk 1 (371 chars) ---
A1C (also called: Glycohemoglobin, HbA1C, Hemoglobin A1C test): Your A1C test result is given in percentages. The higher the percentage, the higher your blood sugar levels have been:  A normal A1C level is below 5.7%  Prediabetes is between 5.7 to 6.

--- chunk 2 (517 chars) ---
A1C (also called: Glycohemoglobin, HbA1C, Hemoglobin A1C test): Prediabetes is between 5.7 to 6.4%. Having prediabetes is a risk factor for getting type 2 diabetes. People with prediabetes may need retests every year.  Type 2 diabetes is above 6.5%  

--- chunk 3 (473 chars) ---
A1C (also called: Glycohemoglobin, HbA1C, Hemoglobin A1C test): If you have diabetes, you should have the A1

## Build the index

Embedding model is BAAI/bge-small-en-v1.5. Open source, small enough to run on a laptop.

Deliberately not using a biomedical model like MedCPT here - those are trained on PubMed abstracts,
which is research writing, and MedlinePlus is written for patients in plain English. So a general
model should be closer to the language we've actually got. That's an assumption though, not
something I've read in a paper, so the plan is to rerun this with a second model later and compare
properly.

Chroma for the vector store, cosine similarity. Zhao et al used Chroma for basically this same task.

This cell is the slow one but it only runs once, after that it loads off disk.

In [ ]:
model = SentenceTransformer(EMBED_MODEL)
client = chromadb.PersistentClient(path=DB_DIR)

REBUILD = False

existing = [c.name for c in client.list_collections()]

if COLLECTION in existing and not REBUILD:
    coll = client.get_collection(COLLECTION)
    print("loaded existing index:", coll.count(), "chunks")
else:
    if COLLECTION in existing:
        client.delete_collection(COLLECTION)
    coll = client.create_collection(COLLECTION, metadata={'hnsw:space': 'cosine'})

    ids, docs, metas = [], [], []
    for t in topics:
        for i, chunk in enumerate(chunks_by_topic[t['id']]):
            ids.append(f"{t['id']}_{i}")
            docs.append(chunk)
            metas.append({
                'topic_id': t['id'],
                'title': t['title'],
                'url': t['url'],
                'group': primary_group(t),
                # chroma metadata has to be scalars so the full list gets joined,
                # keeping it because a topic can legitimately sit in several groups
                'all_groups': '; '.join(t['groups']),
            })

    print("embedding", len(docs), "chunks, this takes a few minutes...")
    batch = 256
    for start in range(0, len(docs), batch):
        emb = model.encode(docs[start:start + batch], show_progress_bar=False)
        coll.add(
            ids=ids[start:start + batch],
            documents=docs[start:start + batch],
            metadatas=metas[start:start + batch],
            embeddings=emb.tolist(),
        )
        print(f"  {min(start + batch, len(docs))}/{len(docs)}")

    print("index built:", coll.count(), "chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6438.44it/s]


embedding 10337 chunks, this takes a few minutes...
  256/10337
  512/10337
  768/10337
  1024/10337
  1280/10337
  1536/10337
  1792/10337
  2048/10337
  2304/10337
  2560/10337
  2816/10337
  3072/10337
  3328/10337
  3584/10337
  3840/10337
  4096/10337
  4352/10337
  4608/10337
  4864/10337
  5120/10337
  5376/10337
  5632/10337
  5888/10337
  6144/10337
  6400/10337
  6656/10337
  6912/10337
  7168/10337
  7424/10337
  7680/10337
  7936/10337
  8192/10337
  8448/10337
  8704/10337
  8960/10337
  9216/10337
  9472/10337
  9728/10337
  9984/10337
  10240/10337
  10337/10337
index built: 10337 chunks


## Retrieval

First version just took the top 5 chunks straight out of Chroma. Ran it and hit an obvious problem -
for "high blood sugar diet" all 5 results were chunks of the same Diabetic Diet topic, and the heart
failure query gave 4 chunks of Heart Failure. Makes sense (a topic's chunks all look similar to a
query about that topic) but it wastes the retrieval budget and would stuff the LLM's context with
near duplicates next week.

So: pull a bigger pool of candidates, then let each topic contribute at most MAX_PER_TOPIC chunks,
working down the list until we have k. Still ranked by similarity, just spread across more topics.

In [103]:
def embed_query(query):
    # bge models were trained with this instruction on the query side only,
    # documents get embedded plain. leaving it off makes retrieval noticeably worse
    return model.encode("Represent this sentence for searching relevant passages: " + query)


def retrieve(query, k=TOP_K, max_per_topic=MAX_PER_TOPIC):
    res = coll.query(query_embeddings=[embed_query(query).tolist()], n_results=k * 10)

    hits = []
    spillover = []          # chunks skipped by the per-topic cap, still ranked
    per_topic = Counter()

    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        hit = {
            'text': doc,
            'title': meta['title'],
            'url': meta['url'],
            'group': meta['group'],
            'similarity': round(1 - dist, 3),
        }
        if per_topic[meta['topic_id']] >= max_per_topic:
            spillover.append(hit)
            continue
        per_topic[meta['topic_id']] += 1
        hits.append(hit)
        if len(hits) == k:
            break

    # if the cap starved us, top up from the best rejected chunks
    if len(hits) < k:
        hits.extend(spillover[:k - len(hits)])

    return hits


def show(query, k=TOP_K):
    print("=" * 70)
    print("QUERY:", query)
    for h in retrieve(query, k=k):
        print(f"  [{h['similarity']}] {h['title']}  ({h['group']})")
        print("      ", h['text'][:100].replace('\n', ' '), "...")
    print()

In [104]:
# checking the dedup did what I wanted - this query returned 5x Diabetic Diet before
print("WITHOUT dedup:")
for h in retrieve('high blood sugar diet', max_per_topic=99):
    print("  ", h['similarity'], h['title'])

print("\nWITH dedup:")
for h in retrieve('high blood sugar diet'):
    print("  ", h['similarity'], h['title'])

WITHOUT dedup:
   0.831 Diabetic Diet
   0.818 Diabetic Diet
   0.804 Diabetic Diet
   0.803 Diabetic Diet
   0.802 Diabetic Diet

WITH dedup:
   0.831 Diabetic Diet
   0.818 Diabetic Diet
   0.795 Hyperglycemia
   0.787 Hyperglycemia
   0.766 Hypoglycemia


## Manual retrieval checks

This is the actual week 3 deliverable. The queries are deliberately mixed: some phrased like a
discharge question, some describing symptoms, some using lay words that don't match the topic title,
and one vague one to see what happens.

For each one the question is just: is the topic I'd expect somewhere in the top 5? Anything that
misses goes in the log at the bottom and feeds the error analysis later.

In [105]:
test_queries = [
    # phrased like someone just discharged
    'What should I do after being discharged with heart failure?',
    'how do I take care of a surgical wound at home',
    'when can I exercise after a hip replacement',
    # symptoms, everyday wording
    'my child has a fever and a rash, what could it be',
    'I feel sad all the time and cannot sleep',
    'trouble breathing at night',
    # lay terms that don't match the topic titles
    'high blood sugar diet',
    'blood thinner side effects',
    # deliberately vague
    'is my medicine safe',
]

for q in test_queries:
    hits = retrieve(q)
    assert len(hits) == TOP_K, f"only got {len(hits)} hits for: {q}"
    show(q)

QUERY: What should I do after being discharged with heart failure?
  [0.698] Heart Attack  (Blood, Heart and Circulation)
       Heart Attack (also called: MI, Myocardial infarction): At the hospital, health care providers make a ...
  [0.693] Cardiac Rehabilitation  (Blood, Heart and Circulation)
       Cardiac Rehabilitation: A heart valve repair or replacement  A heart transplant or a lung transplant ...
  [0.69] Heart Failure  (Blood, Heart and Circulation)
       Heart Failure (also called: CHF, Cardiac failure, Congestive heart failure, Left-sided heart failure ...
  [0.685] Heart Failure  (Blood, Heart and Circulation)
       Heart Failure (also called: CHF, Cardiac failure, Congestive heart failure, Left-sided heart failure ...
  [0.681] Respiratory Failure  (Lungs and Breathing)
       Respiratory Failure: Acute respiratory failure can be a medical emergency. You may need treatment in ...

QUERY: how do I take care of a surgical wound at home
  [0.729] Knee Replacement  (Bones

In [106]:
# following up the two that looked wrong on the first run.
# both of these topics definitely exist in the corpus, so if they don't come back
# it's a retrieval problem not a coverage problem
titles = [t['title'] for t in topics]
for name in ['Wounds and Injuries', 'After Surgery', 'Depression']:
    print(name, "in corpus:", name in titles)

print()
show('how do I take care of a surgical wound at home')
show('I feel sad all the time and cannot sleep')

Wounds and Injuries in corpus: True
After Surgery in corpus: True
Depression in corpus: True

QUERY: how do I take care of a surgical wound at home
  [0.729] Knee Replacement  (Bones, Joints and Muscles)
       Knee Replacement (also called: Knee arthroplasty): Some people go home the same day they have surger ...
  [0.691] First Aid  (Injuries and Wounds)
       First Aid: You can also learn to handle common injuries and wounds . Cuts and scrapes, for example,  ...
  [0.68] Sepsis  (Infections)
       Sepsis: Treating the source of the infection.  If needed, medicines to increase blood pressure.  In  ...
  [0.678] Cesarean Delivery  (Pregnancy and Reproduction)
       Cesarean Delivery (also called: C-section, Cesarean section): Make a cut in your abdomen and uterus. ...
  [0.676] MRSA  (Infections)
       MRSA (also called: Methicillin-resistant Staphylococcus aureus): Wash soiled sheets, towels, and clo ...

QUERY: I feel sad all the time and cannot sleep
  [0.707] Insomnia  (Brain 

In [107]:
# rephrasing the two failures to see whether it's the wording that throws it off
show('wound care')
show('depression')
show('feeling depressed and hopeless')

QUERY: wound care
  [0.755] Wounds and Injuries  (Skin, Hair and Nails)
       Wounds and Injuries (also called: Traumatic injuries): Wounds are injuries that break the skin or ot ...
  [0.717] Diabetic Foot  (Blood, Heart and Circulation)
       Diabetic Foot: Having an infection and poor blood flow can lead to gangrene . That means the muscle, ...
  [0.714] First Aid  (Injuries and Wounds)
       First Aid: You can also learn to handle common injuries and wounds . Cuts and scrapes, for example,  ...
  [0.707] Diabetic Foot  (Blood, Heart and Circulation)
       Diabetic Foot: If you can't feel pain, you may not know when you have a cut, blister , or ulcer (ope ...
  [0.702] Sepsis  (Infections)
       Sepsis: Treating the source of the infection.  If needed, medicines to increase blood pressure.  In  ...

QUERY: depression
  [0.813] Depression  (Mental Health and Behavior)
       Depression (also called: Clinical depression, Dysthymic disorder, Major depressive disorder, Unipola ...


## Does the gold subset retrieve at all

Crude check but a useful one - search each gold topic by its own title and see whether it comes back
in the top 5. If a topic can't even find itself, something is wrong with how it got indexed.

In [108]:
with open('gold_subset.json', encoding='utf-8') as f:
    gold = json.load(f)

misses = []
for g in gold:
    found = [h['title'] for h in retrieve(g['title'])]
    if g['title'] not in found:
        misses.append((g['title'], found[:2]))

print(f"gold self retrieval: {len(gold) - len(misses)}/{len(gold)} found themselves in top {TOP_K}")
for title, got in misses:
    print(f"  MISSED {title} -> got {got}")

gold self retrieval: 30/30 found themselves in top 5


## Quick look at k

Not the real experiment - the proper k sweep happens during evaluation once there are metrics to
compare. Just want to see roughly how far down the list stays relevant before it drifts.

In [109]:
for k in [3, 5, 7]:
    print(f"--- k = {k} ---")
    for h in retrieve('when can I exercise after a hip replacement', k=k):
        print(f"  [{h['similarity']}] {h['title']}")
    print()

--- k = 3 ---
  [0.795] Hip Replacement
  [0.782] Hip Replacement
  [0.745] Knee Replacement

--- k = 5 ---
  [0.795] Hip Replacement
  [0.782] Hip Replacement
  [0.745] Knee Replacement
  [0.732] After Surgery
  [0.717] Exercise and Physical Fitness

--- k = 7 ---
  [0.795] Hip Replacement
  [0.782] Hip Replacement
  [0.745] Knee Replacement
  [0.732] After Surgery
  [0.717] Exercise and Physical Fitness
  [0.713] Knee Replacement
  [0.711] How Much Exercise Do I Need?

